In [2]:
import math
import warnings
from pathlib import Path
from dataclasses import dataclass
from typing import Literal

import numpy as np
import numpy.typing as npt
import pandas as pd
import plotly.graph_objects as go
import random
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

warnings.filterwarnings("ignore")

In [3]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [4]:
DATA_DIR = Path("../data")
CHECKPOINT_DIR = Path("./checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)
DEVICE = (
    torch.device("cuda") if torch.cuda.is_available() else 
    torch.device("mps") if torch.backends.mps.is_available() else
    torch.device("cpu")
)
print(f"Device: {DEVICE}")

Device: mps


## Data Loading

In [5]:
class WindowDataset(Dataset):
    def __init__(
        self,
        df: pd.DataFrame,
        feature_cols: list[str],
        seq_len: int = 20,
        target_col: str = "target",
    ):
        self.seq_len = seq_len
        self.samples: list[tuple[npt.NDArray[np.float32], np.float32]] = []

        for ticker, group in df.groupby("ticker"):
            group = group.sort_index()
            X = group[feature_cols].values.astype(np.float32)
            y = group[target_col].values.astype(np.float32)

            mask = np.isfinite(X).all(axis=1) & np.isfinite(y)
            X = X[mask]
            y = y[mask]

            for i in range(seq_len, len(group)):
                window = X[i - seq_len : i]
                target = y[i]
                self.samples.append((window, target))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        x, y = self.samples[idx]
        return torch.from_numpy(x), torch.tensor(y)

In [6]:
def make_loaders(
    train_df: pd.DataFrame,
    val_df: pd.DataFrame,
    test_df: pd.DataFrame,
    feature_cols: list[str],
    seq_len: int = 20,
    batch_size: int = 64,
) -> tuple[DataLoader, DataLoader, DataLoader]:
    kw = dict(feature_cols=feature_cols, seq_len=seq_len)
    train_ds = WindowDataset(train_df, **kw)
    val_ds = WindowDataset(val_df, **kw)
    test_ds = WindowDataset(test_df, **kw)

    loader_kw = dict(batch_size=batch_size, num_workers=0)
    return (
        DataLoader(train_ds, shuffle=True, **loader_kw),
        DataLoader(val_ds, shuffle=False, **loader_kw),
        DataLoader(test_ds, shuffle=False, **loader_kw),
    )

## Model

### Two-Head GRU with Attention

The backbone (GRU + attention) produces a shared representation.
Two independent heads decode it:

- **Direction head** → P(return > 0) via BCE loss  
- **Magnitude head** → E[|return|] via Huber loss  

Final prediction = soft_sign(direction) × magnitude

In [7]:
class AdditiveAttention(nn.Module):
    """
    Bahdanau-style attention over the time dimension.

    Given hidden states H  (B, T, H), produces a context vector (B, H)
    that is a weighted sum of all timesteps.

    Score:  e_t = v · tanh(W · h_t)      (learned scalar per timestep)
    Weight: α   = softmax(e)              (over T)
    Output: c   = Σ α_t * h_t
    """

    def __init__(self, hidden_size: int):
        super().__init__()
        self.W = nn.Linear(hidden_size, hidden_size, bias=True)
        self.v = nn.Linear(hidden_size, 1, bias=False)

    def forward(self, hidden_states: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
        scores = self.v(torch.tanh(self.W(hidden_states)))  # (B, T, 1)
        weights = torch.softmax(scores, dim=1)               # (B, T, 1)
        context = (weights * hidden_states).sum(dim=1)        # (B, H)
        return context, weights.squeeze(-1)

In [8]:
class TwoHeadGRU(nn.Module):
    """
    Shared GRU + Attention backbone with separate direction and magnitude heads.
    
    Direction head → P(return > 0) via sigmoid  (trained with BCE)
    Magnitude head → E[|return|]   via softplus (trained with Huber)
    Combined       → soft_sign × magnitude
    """
    
    def __init__(
        self,
        input_size: int,
        hidden_size: int = 32,
        num_layers: int = 2,
        head_hidden: int = 16,
        dropout: float = 0.2,
    ):
        super().__init__()
        
        self.gru = nn.GRU(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.attention = AdditiveAttention(hidden_size)
        self.norm = nn.LayerNorm(hidden_size)
        
        # Direction head: context → P(up)
        self.dir_head = nn.Sequential(
            nn.Linear(hidden_size, head_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden, 1),
            # No sigmoid — use BCEWithLogitsLoss for numerical stability
        )
        
        # Magnitude head: context → E[|return|]
        self.mag_head = nn.Sequential(
            nn.Linear(hidden_size, head_hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(head_hidden, 1),
            nn.Softplus(),  # Ensures positive output
        )
        
        self._init_weights()
    
    def _init_weights(self):
        for name, p in self.named_parameters():
            if 'weight_ih' in name:
                nn.init.xavier_uniform_(p)
            elif 'weight_hh' in name:
                nn.init.orthogonal_(p)
            elif 'bias' in name and 'gru' in name:
                nn.init.zeros_(p)
    
    def forward(self, x):
        """
        Returns
        -------
        dir_logit : (B, 1) — raw logit for P(up)
        magnitude : (B, 1) — predicted |return|, always > 0
        combined  : (B,)   — signed prediction for backward compat
        attn      : (B, T) — attention weights
        """
        gru_out, _ = self.gru(x)                           # (B, T, H)
        context, attn = self.attention(gru_out)             # (B, H), (B, T)
        context = self.norm(context)
        
        dir_logit = self.dir_head(context)                  # (B, 1)
        magnitude = self.mag_head(context)                  # (B, 1)
        
        # Combined prediction: soft sign × magnitude
        dir_prob = torch.sigmoid(dir_logit)
        sign = 2.0 * dir_prob - 1.0                        # [-1, 1]
        combined = (sign * magnitude).squeeze(-1)           # (B,)
        
        return dir_logit, magnitude, combined, attn

In [9]:
class TwoHeadLoss(nn.Module):
    """
    Combined loss: α · BCE(direction) + (1-α) · Huber(magnitude)
    
    Parameters
    ----------
    alpha : weight for direction loss. Start at 0.7 (direction-dominant).
        If dir_acc drops when lowering alpha, magnitude gradients are
        corrupting the backbone — raise alpha back up.
    huber_delta : delta for Huber loss on magnitude head.
    """
    
    def __init__(self, alpha: float = 0.7, huber_delta: float = 1.0):
        super().__init__()
        self.alpha = alpha
        self.bce = nn.BCEWithLogitsLoss()
        self.huber = nn.HuberLoss(delta=huber_delta)
    
    def forward(self, dir_logit, magnitude, target):
        """
        Parameters
        ----------
        dir_logit : (B, 1) raw logits from direction head
        magnitude : (B, 1) predicted |return|
        target    : (B,)   actual z-scored return (signed)
        """
        target_2d = target.unsqueeze(-1) if target.dim() == 1 else target
        
        # Direction: binary target
        target_sign = (target_2d > 0).float()
        dir_loss = self.bce(dir_logit, target_sign)
        
        # Magnitude: predict absolute value
        target_mag = target_2d.abs()
        mag_loss = self.huber(magnitude, target_mag)
        
        combined_loss = self.alpha * dir_loss + (1 - self.alpha) * mag_loss
        return combined_loss, dir_loss, mag_loss

## Training Loop

In [10]:
def train_one_epoch(
    model: nn.Module,
    loader: DataLoader,
    optimizer: torch.optim.Optimizer,
    criterion: TwoHeadLoss,
    device: torch.device,
    clip_grad: float = 1.0,
) -> dict:
    model.train()
    total_loss = 0.0
    total_dir  = 0.0
    total_mag  = 0.0
    n = 0

    for x, y in loader:
        x, y = x.to(device), y.to(device)

        optimizer.zero_grad()
        dir_logit, magnitude, combined, _ = model(x)
        loss, dir_loss, mag_loss = criterion(dir_logit, magnitude, y)
        loss.backward()

        if clip_grad > 0:
            nn.utils.clip_grad_norm_(model.parameters(), clip_grad)

        optimizer.step()
        
        bs = len(y)
        total_loss += loss.item() * bs
        total_dir  += dir_loss.item() * bs
        total_mag  += mag_loss.item() * bs
        n += bs

    return {
        "loss": total_loss / n,
        "dir_loss": total_dir / n,
        "mag_loss": total_mag / n,
    }

In [11]:
@torch.no_grad()
def evaluate(
    model: nn.Module,
    loader: DataLoader,
    criterion: TwoHeadLoss,
    device: torch.device,
) -> dict:
    """Evaluate both heads. Returns comprehensive metrics dict."""
    model.eval()
    all_dir, all_mag, all_combined, all_targets = [], [], [], []

    for x, y in loader:
        x, y = x.to(device), y.to(device)
        dir_logit, magnitude, combined, _ = model(x)
        all_dir.append(dir_logit.cpu())
        all_mag.append(magnitude.cpu())
        all_combined.append(combined.cpu())
        all_targets.append(y.cpu())

    dir_logits = torch.cat(all_dir)
    magnitudes = torch.cat(all_mag)
    combined   = torch.cat(all_combined)
    targets    = torch.cat(all_targets)

    # Losses
    loss, dir_loss, mag_loss = criterion(dir_logits, magnitudes, targets)

    # Direction accuracy
    pred_up   = (dir_logits.squeeze(-1) > 0).float()
    actual_up = (targets > 0).float()
    dir_acc   = (pred_up == actual_up).float().mean().item()

    # Direction confidence: 0 = random, 1 = certain
    dir_prob = torch.sigmoid(dir_logits.squeeze(-1))
    dir_confidence = (dir_prob - 0.5).abs().mean().item() * 2

    # Magnitude MAE
    mag_mae = (magnitudes.squeeze(-1) - targets.abs()).abs().mean().item()

    # Combined prediction correlation with target
    t, c = targets, combined
    if t.std() > 0 and c.std() > 0:
        corr = torch.corrcoef(torch.stack([t, c]))[0, 1].item()
    else:
        corr = 0.0

    # Combined MAE and dir_acc (for backward compat comparisons)
    combined_mae = (combined - targets).abs().mean().item()

    return {
        "loss":           loss.item(),
        "dir_loss":       dir_loss.item(),
        "mag_loss":       mag_loss.item(),
        "dir_acc":        dir_acc,
        "dir_confidence": dir_confidence,
        "mag_mae":        mag_mae,
        "combined_mae":   combined_mae,
        "pred_target_corr": corr,
    }

In [12]:
def train(
    model:        nn.Module,
    train_loader: DataLoader,
    val_loader:   DataLoader,
    device:       torch.device,
    lr:           float = 1e-3,
    epochs:       int   = 1000,
    patience:     int   = 20,
    alpha:        float = 0.7,
    huber_delta:  float = 1.0,
    weight_decay: float = 1e-2,
    warmup_epochs: int  = 10,
) -> nn.Module:
    """
    Full training loop for two-head model.
    
    - TwoHeadLoss with configurable alpha
    - AdamW with linear warmup → cosine decay
    - Early stopping on combined val loss
    """
    criterion = TwoHeadLoss(alpha=alpha, huber_delta=huber_delta)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    # Linear warmup then cosine decay
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * progress))
    
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    
    best_val_loss = float("inf")
    patience_ctr  = 0
    best_state    = None

    for epoch in range(1, epochs + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device)
        val_metrics   = evaluate(model, val_loader, criterion, device)
        scheduler.step()

        print(
            f"  Epoch {epoch:3d}  "
            f"train_loss={train_metrics['loss']:.4f}  "
            f"val_loss={val_metrics['loss']:.4f}  "
            f"dir_acc={val_metrics['dir_acc']:.3f}  "
            f"dir_conf={val_metrics['dir_confidence']:.3f}  "
            f"mag_mae={val_metrics['mag_mae']:.3f}  "
            f"corr={val_metrics['pred_target_corr']:.3f}"
        )

        if val_metrics["loss"] < best_val_loss:
            best_val_loss = val_metrics["loss"]
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                print(f"  Early stop at epoch {epoch}.")
                break

    model.load_state_dict(best_state)
    return model

## Cross-Validation (Two-Head Compatible)

In [13]:
from timeseries_cv import (
    expanding_window_folds, sliding_window_folds,
    apply_fold, CVFold,
)

@dataclass
class FoldResult:
    fold_idx:         int
    strategy:         str
    train_loss:       float
    val_loss:         float
    val_dir_loss:     float
    val_mag_loss:     float
    val_dir_acc:      float
    val_dir_conf:     float
    val_mag_mae:      float
    val_corr:         float
    n_train:          int
    n_val:            int
    train_start:      str
    train_end:        str
    val_start:        str
    val_end:          str


def cross_validate(
    full_df:        pd.DataFrame,
    feature_cols:   list[str],
    model_factory,                        # callable() -> nn.Module
    dataset_cls:    type[Dataset],
    strategy:       Literal["expanding", "sliding"] = "expanding",
    n_folds:        int   = 5,
    train_frac:     float = 0.40,
    val_frac:       float = 0.10,
    gap_days:       int   = 0,
    seq_len:        int   = 20,
    batch_size:     int   = 64,
    lr:             float = 1e-3,
    epochs:         int   = 1000,
    patience:       int   = 20,
    alpha:          float = 0.7,
    huber_delta:    float = 1.0,
    weight_decay:   float = 1e-2,
    warmup_epochs:  int   = 10,
    device:         torch.device = torch.device("cpu"),
    verbose:        bool  = True,
) -> list[FoldResult]:
    """Time-series CV for two-head models."""
    
    if strategy == "expanding":
        folds = expanding_window_folds(
            full_df, n_folds=n_folds,
            min_train_frac=train_frac, val_frac=val_frac, gap_days=gap_days,
        )
    elif strategy == "sliding":
        folds = sliding_window_folds(
            full_df, n_folds=n_folds,
            train_frac=train_frac, val_frac=val_frac, gap_days=gap_days,
        )
    else:
        raise ValueError(f"Unknown strategy: {strategy!r}")

    criterion = TwoHeadLoss(alpha=alpha, huber_delta=huber_delta)
    results: list[FoldResult] = []

    for fold in folds:
        if verbose:
            print(f"\n{'='*60}")
            print(f"  [{fold.strategy.upper()}] Fold {fold.fold_idx}  |  "
                  f"train {fold.train_start.date()} → {fold.train_end.date()}  |  "
                  f"val {fold.val_start.date()} → {fold.val_end.date()}")
            print(f"{'='*60}")

        train_df, val_df = apply_fold(full_df, fold)
        train_ds = dataset_cls(train_df, feature_cols=feature_cols, seq_len=seq_len)
        val_ds   = dataset_cls(val_df,   feature_cols=feature_cols, seq_len=seq_len)

        if len(train_ds) == 0 or len(val_ds) == 0:
            print(f"  ⚠ Fold {fold.fold_idx} skipped (empty dataset).")
            continue

        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0)
        val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0)

        model = model_factory()
        optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
        
        def lr_lambda(epoch):
            if epoch < warmup_epochs:
                return (epoch + 1) / warmup_epochs
            progress = (epoch - warmup_epochs) / max(1, epochs - warmup_epochs)
            return 0.5 * (1 + math.cos(math.pi * progress))
        
        scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
        
        best_val_loss = float("inf")
        patience_ctr  = 0
        best_state    = None
        best_metrics  = None

        for epoch in range(1, epochs + 1):
            t_metrics = train_one_epoch(model, train_loader, optimizer, criterion, device)
            v_metrics = evaluate(model, val_loader, criterion, device)
            scheduler.step()

            if verbose:
                print(f"  Epoch {epoch:3d}  "
                      f"train_loss={t_metrics['loss']:.4f}  "
                      f"val_loss={v_metrics['loss']:.4f}  "
                      f"dir_acc={v_metrics['dir_acc']:.3f}  "
                      f"corr={v_metrics['pred_target_corr']:.3f}")

            if v_metrics["loss"] < best_val_loss:
                best_val_loss = v_metrics["loss"]
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
                best_metrics = (t_metrics, v_metrics)
                patience_ctr = 0
            else:
                patience_ctr += 1
                if patience_ctr >= patience:
                    if verbose:
                        print(f"  Early stop at epoch {epoch}.")
                    break

        tm, vm = best_metrics
        results.append(FoldResult(
            fold_idx      = fold.fold_idx,
            strategy      = fold.strategy,
            train_loss    = tm["loss"],
            val_loss      = vm["loss"],
            val_dir_loss  = vm["dir_loss"],
            val_mag_loss  = vm["mag_loss"],
            val_dir_acc   = vm["dir_acc"],
            val_dir_conf  = vm["dir_confidence"],
            val_mag_mae   = vm["mag_mae"],
            val_corr      = vm["pred_target_corr"],
            n_train       = len(train_ds),
            n_val         = len(val_ds),
            train_start   = str(fold.train_start.date()),
            train_end     = str(fold.train_end.date()),
            val_start     = str(fold.val_start.date()),
            val_end       = str(fold.val_end.date()),
        ))

    return results


def summarize_cv(results: list[FoldResult]) -> pd.DataFrame:
    if not results:
        print("No results.")
        return pd.DataFrame()

    strategy = results[0].strategy
    df = pd.DataFrame([vars(r) for r in results])

    print(f"\n{'='*70}")
    print(f"  {strategy.upper()}-WINDOW CV SUMMARY")
    print(f"{'='*70}")
    for r in results:
        print(f"  Fold {r.fold_idx}  |  val_loss={r.val_loss:.4f}  "
              f"dir_acc={r.val_dir_acc:.3f}  dir_conf={r.val_dir_conf:.3f}  "
              f"mag_mae={r.val_mag_mae:.3f}  corr={r.val_corr:.3f}  |  "
              f"train={r.n_train:,}  val={r.n_val:,}")

    print(f"\n  val_loss   : {df['val_loss'].mean():.4f} ± {df['val_loss'].std():.4f}")
    print(f"  dir_acc    : {df['val_dir_acc'].mean():.3f} ± {df['val_dir_acc'].std():.3f}")
    print(f"  dir_conf   : {df['val_dir_conf'].mean():.3f} ± {df['val_dir_conf'].std():.3f}")
    print(f"  mag_mae    : {df['val_mag_mae'].mean():.3f} ± {df['val_mag_mae'].std():.3f}")
    print(f"  corr       : {df['val_corr'].mean():.3f} ± {df['val_corr'].std():.3f}")
    print(f"{'='*70}")
    return df

In [14]:
@dataclass
class EvalResult:
    eval_loss:    float
    eval_mae:     float
    dir_acc:      float
    dir_conf:     float
    mag_mae:      float
    corr:         float
    n_train:      int
    n_eval:       int
    eval_dates:   list
    eval_preds:   list
    eval_targets: list
    eval_dir_probs: list
    eval_magnitudes: list
    per_ticker:   pd.DataFrame


def evaluate_holdout(
    train_df:      pd.DataFrame,
    eval_df:       pd.DataFrame,
    feature_cols:  list[str],
    model_factory,
    dataset_cls:   type[Dataset],
    lr:            float = 1e-3,
    epochs:        int   = 1000,
    patience:      int   = 20,
    alpha:         float = 0.7,
    huber_delta:   float = 1.0,
    weight_decay:  float = 1e-2,
    warmup_epochs: int   = 10,
    seq_len:       int   = 20,
    batch_size:    int   = 64,
    device:        torch.device = torch.device("cpu"),
    ticker:        list[str] | None = None,
    verbose:       bool = True,
) -> EvalResult:
    """Train on train_df, evaluate on eval_df. Optionally filter by ticker."""
    
    if ticker is not None:
        eval_df = eval_df[eval_df["ticker"].isin(ticker)]
    
    # Use last 10% of train as internal val for early stopping
    dates = pd.DatetimeIndex(train_df.index).sort_values().unique()
    split_date = dates[int(len(dates) * 0.9)]
    internal_train = train_df.loc[train_df.index <= split_date]
    internal_val   = train_df.loc[train_df.index > split_date]
    
    kw = dict(feature_cols=feature_cols, seq_len=seq_len)
    train_ds = dataset_cls(internal_train, **kw)
    val_ds   = dataset_cls(internal_val,   **kw)
    eval_ds  = dataset_cls(eval_df,        **kw)
    
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,  num_workers=0)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False, num_workers=0)
    eval_loader  = DataLoader(eval_ds,  batch_size=batch_size, shuffle=False, num_workers=0)
    
    tickers_str = ", ".join(ticker) if ticker else "all"
    n_train = len(train_ds) + len(val_ds)
    n_eval  = len(eval_ds)
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"  HOLDOUT EVALUATION  |  ticker={tickers_str}")
        print(f"  train {train_df.index.min().date()} -> {train_df.index.max().date()}  "
              f"({n_train:,} rows)")
        print(f"  eval  {eval_df.index.min().date()} -> {eval_df.index.max().date()}  "
              f"({n_eval:,} rows)")
        print(f"{'='*60}")
    
    # Train
    criterion = TwoHeadLoss(alpha=alpha, huber_delta=huber_delta)
    model = model_factory()
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    
    def lr_lambda(epoch):
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs
        progress = (epoch - warmup_epochs) / max(1, epochs - warmup_epochs)
        return 0.5 * (1 + math.cos(math.pi * progress))
    
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    
    best_val_loss = float("inf")
    patience_ctr = 0
    best_state = None
    best_epoch = 0
    
    for epoch in range(1, epochs + 1):
        t_m = train_one_epoch(model, train_loader, optimizer, criterion, device)
        v_m = evaluate(model, val_loader, criterion, device)
        e_m = evaluate(model, eval_loader, criterion, device)
        scheduler.step()
        
        if verbose:
            print(f"  Epoch {epoch:3d}  "
                  f"train_loss={t_m['loss']:.4f}  "
                  f"eval_loss={e_m['loss']:.4f}  "
                  f"dir_acc={e_m['dir_acc']:.3f}  "
                  f"corr={e_m['pred_target_corr']:.3f}")
        
        if v_m["loss"] < best_val_loss:
            best_val_loss = v_m["loss"]
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            best_epoch = epoch
            patience_ctr = 0
        else:
            patience_ctr += 1
            if patience_ctr >= patience:
                if verbose:
                    print(f"  Early stop at epoch {epoch}.")
                break
    
    model.load_state_dict(best_state)
    
    # Final eval — collect per-sample predictions
    model.eval()
    all_dir, all_mag, all_combined, all_targets = [], [], [], []
    with torch.no_grad():
        for x, y in eval_loader:
            x, y = x.to(device), y.to(device)
            dl, mg, cb, _ = model(x)
            all_dir.append(torch.sigmoid(dl).cpu())
            all_mag.append(mg.cpu())
            all_combined.append(cb.cpu())
            all_targets.append(y.cpu())
    
    dir_probs  = torch.cat(all_dir).squeeze(-1).numpy()
    magnitudes = torch.cat(all_mag).squeeze(-1).numpy()
    combined   = torch.cat(all_combined).numpy()
    targets    = torch.cat(all_targets).numpy()
    
    # Metrics
    dir_acc = ((combined > 0) == (targets > 0)).mean()
    mae     = np.abs(combined - targets).mean()
    corr    = np.corrcoef(combined, targets)[0, 1] if targets.std() > 0 else 0.0
    dir_conf = np.abs(dir_probs - 0.5).mean() * 2
    mag_mae  = np.abs(magnitudes - np.abs(targets)).mean()
    
    dummy_loss = criterion(
        torch.zeros(len(targets), 1), torch.ones(len(targets), 1) * np.abs(targets).mean(),
        torch.from_numpy(targets)
    )[0].item()
    
    # Eval dates from the eval_df (offset by seq_len)
    eval_dates = []
    for tk, grp in eval_df.groupby("ticker"):
        grp = grp.sort_index()
        eval_dates.extend(grp.index[seq_len:].tolist())
    eval_dates = eval_dates[:len(targets)]
    
    # Per-ticker breakdown
    per_ticker_rows = []
    if ticker is not None:
        idx = 0
        for tk in ticker:
            tk_df = eval_df[eval_df["ticker"] == tk].sort_index()
            n_tk = len(tk_df) - seq_len
            if n_tk <= 0:
                continue
            tk_preds = combined[idx:idx+n_tk]
            tk_targets = targets[idx:idx+n_tk]
            tk_dir = ((tk_preds > 0) == (tk_targets > 0)).mean()
            tk_corr = np.corrcoef(tk_preds, tk_targets)[0, 1] if tk_targets.std() > 0 else 0.0
            per_ticker_rows.append({
                "ticker": tk, "dir_acc": tk_dir, "corr": tk_corr, "n": n_tk
            })
            idx += n_tk
    
    per_ticker_df = pd.DataFrame(per_ticker_rows) if per_ticker_rows else pd.DataFrame()
    
    if verbose:
        print(f"\n{'='*60}")
        print(f"  HOLDOUT RESULT  |  ticker={tickers_str}")
        print(f"{'='*60}")
        print(f"  dir_acc    : {dir_acc:.3f}")
        print(f"  dir_conf   : {dir_conf:.3f}")
        print(f"  mag_mae    : {mag_mae:.3f}")
        print(f"  corr       : {corr:.3f}")
        print(f"  eval_mae   : {mae:.3f}")
        print(f"  best epoch : {best_epoch}")
        if not per_ticker_df.empty:
            print(f"\n  {'ticker':12s} {'dir_acc':>8s} {'corr':>8s} {'n':>6s}")
            for _, row in per_ticker_df.iterrows():
                print(f"  {row['ticker']:12s} {row['dir_acc']:8.3f} {row['corr']:8.3f} {row['n']:6.0f}")
        print(f"{'='*60}")
    
    return EvalResult(
        eval_loss=0.0, eval_mae=mae, dir_acc=dir_acc, dir_conf=dir_conf,
        mag_mae=mag_mae, corr=corr, n_train=n_train, n_eval=n_eval,
        eval_dates=eval_dates, eval_preds=combined.tolist(),
        eval_targets=targets.tolist(), eval_dir_probs=dir_probs.tolist(),
        eval_magnitudes=magnitudes.tolist(), per_ticker=per_ticker_df,
    )

## Data Loading & Feature Selection

In [15]:
train_df = pd.read_csv("data/train_features.csv", index_col=0, parse_dates=True)
val_df   = pd.read_csv("data/val_features.csv",   index_col=0, parse_dates=True)
test_df  = pd.read_csv("data/test_features.csv",  index_col=0, parse_dates=True)

cv_df   = pd.concat([train_df, val_df]).sort_index()
print(f"CV pool:  {len(cv_df):,} rows  |  "
      f"{cv_df.index.min().date()} → {cv_df.index.max().date()}")
print(f"Test set: {len(test_df):,} rows (held out)")

feature_cols = [
    "bull_regime",           # regime / trend
    "z_bb_pct_b",            # momentum (Bollinger Band position)
    "z_volume_z20",          # volume anomaly
    "z_log_close_return_1",  # yesterday's return
    "z_range",               # volatility
    "z_ret_autocorr",        # trending vs mean-reverting
    "dow_cos",               # calendar
    "high_vol_regime",       # vol regime (binary)
]
print(f"Features ({len(feature_cols)}): {feature_cols}")

CV pool:  14,574 rows  |  2015-04-03 → 2025-05-27
Test set: 1,710 rows (held out)
Features (8): ['bull_regime', 'z_bb_pct_b', 'z_volume_z20', 'z_log_close_return_1', 'z_range', 'z_ret_autocorr', 'dow_cos', 'high_vol_regime']


## Baseline Sanity Check

In [16]:
# Compute always-predict-up baseline and dummy (predict-zero) loss per fold
from timeseries_cv import expanding_window_folds

huber = nn.HuberLoss(delta=1.0)

print("EXPANDING FOLDS — Baselines:")
for f in expanding_window_folds(cv_df, n_folds=5, min_train_frac=0.40, val_frac=0.10):
    targets = cv_df.loc[f.val_start : f.val_end, "target"]
    bull_bias = (targets > 0).mean()
    dummy_loss = huber(
        torch.zeros_like(torch.from_numpy(targets.values)),
        torch.from_numpy(targets.values),
    ).item()
    print(f"  Fold {f.fold_idx}: {f.train_start.date()} → {f.train_end.date()} | "
          f"val {f.val_start.date()} → {f.val_end.date()} | "
          f"bull_bias={bull_bias:.3f}  dummy_loss={dummy_loss:.4f}")

EXPANDING FOLDS — Baselines:
  Fold 0: 2015-04-03 → 2019-04-24 | val 2019-04-25 → 2020-04-28 | bull_bias=0.515  dummy_loss=0.3933
  Fold 1: 2015-04-03 → 2020-07-30 | val 2020-07-31 → 2021-08-04 | bull_bias=0.498  dummy_loss=0.4181
  Fold 2: 2015-04-03 → 2021-11-06 | val 2021-11-07 → 2022-11-11 | bull_bias=0.499  dummy_loss=0.3928
  Fold 3: 2015-04-03 → 2023-02-13 | val 2023-02-14 → 2024-02-18 | bull_bias=0.486  dummy_loss=0.3634
  Fold 4: 2015-04-03 → 2024-05-22 | val 2024-05-23 → 2025-05-27 | bull_bias=0.495  dummy_loss=0.3807


## Model Config & Cross-Validation

In [17]:
SEQ_LEN = 20

def model_factory():
    torch.manual_seed(SEED)
    return TwoHeadGRU(
        input_size=len(feature_cols),
        hidden_size=32,       # more capacity for two heads
        num_layers=2,
        head_hidden=16,
        dropout=0.2,          # lighter reg — let magnitude head express itself
    ).to(DEVICE)

shared_params = dict(
    full_df=cv_df, feature_cols=feature_cols,
    model_factory=model_factory, dataset_cls=WindowDataset,
    n_folds=5, val_frac=0.10,
    seq_len=SEQ_LEN, batch_size=64, lr=1e-3,
    epochs=1000, patience=20,
    alpha=0.7,               # direction-dominant
    huber_delta=1.0,
    weight_decay=1e-2,
    warmup_epochs=10,
    device=DEVICE,
)

In [18]:
exp_results = cross_validate(**shared_params, strategy="expanding")
exp_df = summarize_cv(exp_results)


  [EXPANDING] Fold 0  |  train 2015-04-03 → 2019-04-24  |  val 2019-04-25 → 2020-04-28
  Epoch   1  train_loss=0.5582  val_loss=0.5543  dir_acc=0.514  corr=0.068
  Epoch   2  train_loss=0.5514  val_loss=0.5497  dir_acc=0.514  corr=0.101
  Epoch   3  train_loss=0.5478  val_loss=0.5483  dir_acc=0.515  corr=0.117
  Epoch   4  train_loss=0.5458  val_loss=0.5478  dir_acc=0.531  corr=0.112
  Epoch   5  train_loss=0.5460  val_loss=0.5478  dir_acc=0.522  corr=0.113
  Epoch   6  train_loss=0.5432  val_loss=0.5478  dir_acc=0.509  corr=0.105
  Epoch   7  train_loss=0.5438  val_loss=0.5481  dir_acc=0.520  corr=0.105
  Epoch   8  train_loss=0.5429  val_loss=0.5480  dir_acc=0.516  corr=0.099
  Epoch   9  train_loss=0.5421  val_loss=0.5490  dir_acc=0.509  corr=0.098
  Epoch  10  train_loss=0.5425  val_loss=0.5482  dir_acc=0.511  corr=0.089
  Epoch  11  train_loss=0.5408  val_loss=0.5497  dir_acc=0.514  corr=0.091
  Epoch  12  train_loss=0.5402  val_loss=0.5495  dir_acc=0.513  corr=0.088
  Epoch  13 

In [ ]:
# sli_results = cross_validate(**shared_params, strategy="sliding")
# sli_df = summarize_cv(sli_results)

## Visualization

In [19]:
fold_labels = [f"Fold {r.fold_idx}" for r in exp_results]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=fold_labels,
    y=[r.val_dir_acc for r in exp_results],
    name="Dir. Accuracy", marker_color="#636efa", opacity=0.8,
))
fig.add_trace(go.Bar(
    x=fold_labels,
    y=[r.val_corr for r in exp_results],
    name="Pred-Target Corr", marker_color="#00cc96", opacity=0.8,
))
fig.add_hline(y=0.50, line_dash="dash", line_color="white",
              annotation_text="coin flip")
fig.update_layout(
    template="plotly_dark",
    title="Two-Head GRU — Expanding Window CV",
    yaxis_title="Metric",
    yaxis_range=[0.0, 0.70],
    barmode="group",
)
fig.show()

In [20]:
# Direction confidence across folds — key diagnostic
fig = go.Figure()
fig.add_trace(go.Bar(
    x=fold_labels,
    y=[r.val_dir_conf for r in exp_results],
    name="Dir. Confidence", marker_color="#ef553b", opacity=0.8,
))
fig.update_layout(
    template="plotly_dark",
    title="Direction Head Confidence (0=random, 1=certain)",
    yaxis_title="Confidence",
    yaxis_range=[0.0, 0.5],
)
fig.show()

## Holdout Evaluation

In [21]:
result = evaluate_holdout(
    train_df=cv_df, eval_df=test_df,
    feature_cols=feature_cols,
    model_factory=model_factory,
    dataset_cls=WindowDataset,
    lr=1e-3, device=DEVICE,
    ticker=["BTC-USD", "ETH-USD"],
    alpha=0.7, huber_delta=1.0,
    weight_decay=1e-2, warmup_epochs=10,
)


  HOLDOUT EVALUATION  |  ticker=BTC-USD, ETH-USD
  train 2015-04-03 -> 2025-05-27  (14,334 rows)
  eval  2025-05-28 -> 2026-03-08  (530 rows)
  Epoch   1  train_loss=0.5478  eval_loss=0.5490  dir_acc=0.504  corr=0.053
  Epoch   2  train_loss=0.5441  eval_loss=0.5473  dir_acc=0.536  corr=0.034
  Epoch   3  train_loss=0.5434  eval_loss=0.5476  dir_acc=0.545  corr=0.049
  Epoch   4  train_loss=0.5427  eval_loss=0.5470  dir_acc=0.542  corr=0.029
  Epoch   5  train_loss=0.5428  eval_loss=0.5459  dir_acc=0.545  corr=0.042
  Epoch   6  train_loss=0.5427  eval_loss=0.5459  dir_acc=0.534  corr=0.007
  Epoch   7  train_loss=0.5421  eval_loss=0.5446  dir_acc=0.543  corr=0.006
  Epoch   8  train_loss=0.5425  eval_loss=0.5451  dir_acc=0.534  corr=0.018
  Epoch   9  train_loss=0.5418  eval_loss=0.5463  dir_acc=0.551  corr=0.030
  Epoch  10  train_loss=0.5419  eval_loss=0.5455  dir_acc=0.549  corr=0.038
  Epoch  11  train_loss=0.5406  eval_loss=0.5436  dir_acc=0.551  corr=0.028
  Epoch  12  train_lo

In [22]:
# Predicted vs actual returns
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=result.eval_dates, y=result.eval_targets,
    name="actual", opacity=0.7,
))
fig.add_trace(go.Scatter(
    x=result.eval_dates, y=result.eval_preds,
    name="combined prediction", opacity=0.7,
))
fig.update_layout(template="plotly_dark", title="Holdout: Combined Prediction vs Actual")
fig.show()

In [23]:
# Direction probability over time — shows model conviction
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=result.eval_dates, y=result.eval_dir_probs,
    name="P(up)", mode="lines", opacity=0.7,
))
fig.add_hline(y=0.5, line_dash="dash", line_color="white")

# Color background by actual direction
fig.update_layout(
    template="plotly_dark",
    title="Direction Head: P(up) Over Time",
    yaxis_title="P(return > 0)",
    yaxis_range=[0.0, 1.0],
)
fig.show()

In [24]:
# Magnitude prediction vs actual |return|
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=result.eval_dates, y=[abs(t) for t in result.eval_targets],
    name="|actual|", opacity=0.5,
))
fig.add_trace(go.Scatter(
    x=result.eval_dates, y=result.eval_magnitudes,
    name="predicted magnitude", opacity=0.7,
))
fig.update_layout(template="plotly_dark", title="Magnitude Head: Predicted vs Actual |Return|")
fig.show()